This notebook shows the model-to-fold mapping and checks the **actual input gene TSSs**
for human and mouse.

## 1. Which folds does each model use?


In [3]:
import alphagenome
alphagenome.__version__

'0.7.0'

In [2]:
import pandas as pd
from alphagenome.models import dna_client
from alphagenome.data import fold_intervals

rows = []

for version in dna_client.ModelVersion:
    row = {"model_version": version.name}

    for subset in fold_intervals.Subset:
        # ALL_FOLDS has no held-out validation/test data.
        if version == dna_client.ModelVersion.ALL_FOLDS:
            folds = (
                fold_intervals.get_all_folds()
                if subset == fold_intervals.Subset.TRAIN
                else []
            )
        else:
            folds = fold_intervals.get_fold_names(version, subset)

        row[subset.name.lower()] = ", ".join(folds)

    rows.append(row)

display(pd.DataFrame(rows))

/media/eternus1/nfs/projects/dpanchenko/BenchAG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,model_version,train,valid,test
0,ALL_FOLDS,"fold0, fold1, fold2, fold3, fold4, fold5, fold...",,
1,FOLD_0,"fold2, fold3, fold4, fold5, fold6, fold7",fold0,fold1
2,FOLD_1,"fold0, fold1, fold2, fold5, fold6, fold7",fold3,fold4
3,FOLD_2,"fold0, fold1, fold3, fold4, fold6, fold7",fold2,fold5
4,FOLD_3,"fold0, fold1, fold2, fold3, fold4, fold5",fold6,fold7


## 2. Load the benchmark genes and released Borzoi partitions

Inputs: `data/{human,mouse}.{test,valid}.{forward,reverse}.csv` (tab-separated).
These are the same interval tables used by the expression runner: hg38 for human,
mm10 for mouse. TSS values are treated as zero-based points; BED intervals include
the start and exclude the end.


In [4]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import numpy as np
import requests

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'configs/expression.human.128kb.yaml').exists())
DATA = ROOT / 'data'
FOLD_CACHE = DATA / 'folds'
FOLD_CACHE.mkdir(exist_ok=True)
BORZOI_COMMIT = '5c9358222b5026abb733ed5fb84f3f6c77239b37'
input_hashes = {}
fold_sources = {}
genes = {}
partitions = {}

for organism in ['human', 'mouse']:
    url = f'https://raw.githubusercontent.com/calico/borzoi/{BORZOI_COMMIT}/data/sequences_{organism}.bed.gz'
    path = FOLD_CACHE / f'sequences_{organism}.bed.gz'
    if not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        temporary = path.with_suffix('.tmp')
        temporary.write_bytes(response.content)
        temporary.replace(path)
    partitions[organism] = pd.read_csv(path, sep='\t', header=None,
                                      names=['chrom', 'start', 'end', 'fold'])
    fold_sources[organism] = {'url':url, 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
    for split in ['test', 'valid']:
        pieces = []
        for direction, strand in [('forward', '+'), ('reverse', '-')]:
            path = DATA / f'{organism}.{split}.{direction}.csv'
            frame = pd.read_csv(path, sep='\t', dtype={'gene_id':str, 'chrom':str})
            assert frame.gene_strand.eq(strand).all(), f'Unexpected strand in {path}'
            positions = pd.to_numeric(frame.TSS, errors='raise')
            assert np.isfinite(positions).all() and (positions >= 0).all()
            assert (positions == np.floor(positions)).all(), f'Noninteger TSS in {path}'
            frame['TSS'] = positions.astype('int64')
            pieces.append(frame[['gene_id', 'chrom', 'TSS', 'gene_strand']])
            input_hashes[str(path.relative_to(ROOT))] = hashlib.sha256(path.read_bytes()).hexdigest()
        combined = pd.concat(pieces, ignore_index=True)
        assert combined.gene_id.is_unique, f'Duplicate genes in {organism} {split}'
        genes[organism, split] = combined
    assert not set(genes[organism,'test'].gene_id) & set(genes[organism,'valid'].gene_id)

print(pd.DataFrame([{'organism':o,'split':s,'requested_genes':len(f)}
                    for (o,s),f in genes.items()]).to_string(index=False))


organism split  requested_genes
   human  test             2781
   human valid             3038
   mouse  test             2724
   mouse valid             2998


## 3. Assign every TSS to a fold

For every gene, test `BED start ≤ TSS < BED end` against **all eight folds**


In [5]:
role_by_fold = {}
for subset in fold_intervals.Subset:
    for fold in fold_intervals.get_fold_names(dna_client.ModelVersion.FOLD_1, subset):
        role_by_fold[fold] = subset.name.lower()

assigned = []
for (organism, split), frame in genes.items():
    frame = frame.copy()
    hits = pd.DataFrame(False, index=frame.index, columns=fold_intervals.get_all_folds())
    for chrom in frame.chrom.unique():
        idx = frame.index[frame.chrom == chrom]
        tss = frame.loc[idx, 'TSS'].to_numpy()
        bed = partitions[organism]
        for fold in hits.columns:
            intervals = bed[(bed.chrom == chrom) & (bed.fold == fold)].sort_values('start')
            if intervals.empty:
                continue
            starts = intervals.start.to_numpy()
            ends = np.maximum.accumulate(intervals.end.to_numpy())
            previous = np.searchsorted(starts, tss, side='right') - 1
            hits.loc[idx, fold] = (previous >= 0) & (ends[np.maximum(previous, 0)] > tss)
    frame['matching_folds'] = hits.sum(axis=1)
    frame['borzoi_fold'] = hits.idxmax(axis=1).where(frame.matching_folds == 1, 'UNRESOLVED')
    frame['organism'] = organism
    frame['input_split'] = split
    frame['alphagenome_FOLD_1_subset'] = frame.borzoi_fold.map(role_by_fold)
    frame['expected_fold'] = 'fold3' if split == 'test' else 'fold4'
    frame['matches_expected_fold'] = frame.borzoi_fold == frame.expected_fold
    assigned.append(frame)

per_gene = pd.concat(assigned, ignore_index=True)

display(per_gene[['organism','input_split','gene_id','chrom','TSS','gene_strand',
                  'borzoi_fold','alphagenome_FOLD_1_subset']].head(8))


,organism,input_split,gene_id,chrom,TSS,gene_strand,borzoi_fold,alphagenome_FOLD_1_subset
0,human,test,ENSG00000232721.2,chr1,143736066,+,fold3,valid
1,human,test,ENSG00000263590.2,chr1,143790010,+,fold3,valid
2,human,test,ENSG00000234277.2,chr1,227393591,+,fold3,valid
3,human,test,ENSG00000181450.17,chr1,227563543,+,fold3,valid
4,human,test,ENSG00000143740.14,chr1,227734996,+,fold3,valid
5,human,test,ENSG00000185888.5,chr1,227815693,+,fold3,valid
6,human,test,ENSG00000154342.5,chr1,228007051,+,fold3,valid
7,human,test,ENSG00000143761.15,chr1,228082660,+,fold3,valid


## 4. Verification results

The expected correspondence is Borzoi **test → fold3 → AG validation** and
Borzoi **validation → fold4 → AG test**. The assertions fail if any gene is
unmapped, maps to multiple folds, or belongs to an unexpected fold.


In [7]:
summary = per_gene.groupby(['organism','input_split'], sort=False).agg(
    n_genes=('gene_id','size'),
    borzoi_fold=('borzoi_fold', lambda x: ', '.join(sorted(set(x)))),
    alphagenome_FOLD_1_subset=('alphagenome_FOLD_1_subset', lambda x: ', '.join(sorted(set(x.dropna())))),
    n_expected=('matches_expected_fold','sum'),
    n_unmapped=('matching_folds', lambda x: int((x == 0).sum())),
    n_ambiguous=('matching_folds', lambda x: int((x > 1).sum())),
).reset_index()
summary['n_unexpected_fold'] = summary.n_genes - summary.n_expected

display(summary)

display(pd.crosstab([per_gene.organism, per_gene.input_split], per_gene.borzoi_fold)
        .reindex(columns=fold_intervals.get_all_folds(), fill_value=0))

assert per_gene.matching_folds.eq(1).all(), 'Some TSSs are unmapped or ambiguous'
assert per_gene.matches_expected_fold.all(), 'Some TSSs fall in unexpected folds'
assert role_by_fold['fold3'] == 'valid' and role_by_fold['fold4'] == 'test'

manifest = {'borzoi_commit':BORZOI_COMMIT, 'fold_sources':fold_sources,
            'input_sha256':input_hashes,
            'alphagenome_client':importlib.metadata.version('alphagenome'),
            'coordinate_rule':'zero-based TSS; BED start <= TSS < BED end',
            'model':'FOLD_1', 'summary':summary.to_dict('records')}




,organism,input_split,n_genes,borzoi_fold,alphagenome_FOLD_1_subset,n_expected,n_unmapped,n_ambiguous,n_unexpected_fold
0,human,test,2781,fold3,valid,2781,0,0,0
1,human,valid,3038,fold4,test,3038,0,0,0
2,mouse,test,2724,fold3,valid,2724,0,0,0
3,mouse,valid,2998,fold4,test,2998,0,0,0


borzoi_fold           fold0  fold1  fold2  fold3  fold4  fold5  fold6  fold7
organism input_split                                                        
human    test             0      0      0   2781      0      0      0      0
         valid            0      0      0      0   3038      0      0      0
mouse    test             0      0      0   2724      0      0      0      0
         valid            0      0      0      0   2998      0      0      0